# Primary RQ1 constant sensitivity

Seven registered conditions evaluate ±20% changes in the primary S7 blend, adoption and skip constants on all 136 frozen questions (952 evaluations). The original notebook logic, catalogs and rewrite files are preserved. No generation API key is needed.

Select **Runtime → Change runtime type → A100 GPU**, then **Run all**. Completed score captures and the final ZIP are retained in Drive. This notebook is a launcher; the scientific runner validates its published registration before inference.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = 'a066453a317bba365db707b71745e49fee28811a'
REPO_DIR = Path('/content/ComplianceGPT_sensitivity_v1')
OUTPUT_DIR = Path('/content/drive/MyDrive/ComplianceGPT_runs/rq1_constant_sensitivity_v1')
CONSOLE_LOG = OUTPUT_DIR.parent / 'rq1_constant_sensitivity_v1_console.log'
if len(SOURCE_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in SOURCE_COMMIT):
    raise RuntimeError('Use the finalized public notebook with an immutable source commit.')
if (REPO_DIR / '.git').is_dir():
    dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
    if dirty:
        raise RuntimeError('The local checkout has changes. Use a fresh runtime and retain the Drive outputs.')
else:
    subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip() == SOURCE_COMMIT
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('Registered scientific source:', SOURCE_COMMIT)
print('Persistent result directory:', OUTPUT_DIR)


Registered scientific source: a066453a317bba365db707b71745e49fee28811a
Persistent result directory: /content/drive/MyDrive/ComplianceGPT_runs/rq1_constant_sensitivity_v1


In [3]:
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True, check=True)
print('Allocated GPU:', gpu.stdout.strip())
print('The notebook requests A100; the runner records the actual allocation.')
requirements = REPO_DIR / 'experiments/retriever_ablation/constant_sensitivity/requirements-colab.txt'
# Remove timm with torchvision: an installed timm otherwise advertises a broken optional vision backend.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timm', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)


Allocated GPU: NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
The notebook requests A100; the runner records the actual allocation.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/content/ComplianceGPT_sensitivity_v1/experiments/retriever_ablation/constant_sensitivity/requirements-colab.txt'], returncode=0)

In [4]:
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src') + ':' + str(REPO_DIR)
test_environment['USE_TORCH'] = '1'
test_environment['USE_TF'] = '0'
model_preflight = '''
import importlib.util
unexpected = [name for name in ('timm', 'torchvision', 'torchaudio') if importlib.util.find_spec(name)]
if unexpected:
    raise RuntimeError('Optional media backends remain installed: ' + ', '.join(unexpected))
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModel, AutoModelForSequenceClassification, BertConfig, XLMRobertaConfig
# Tiny models exercise both AutoModel loading paths without downloading or running the study.
common = dict(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16)
AutoModel.from_config(BertConfig(**common))
AutoModelForSequenceClassification.from_config(XLMRobertaConfig(**common))
print('Text-model imports and AutoModel loading preflight passed.')
'''
subprocess.run([sys.executable, '-c', model_preflight], cwd=REPO_DIR, env=test_environment, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_rq1_constant_sensitivity', '-v'],
               cwd=REPO_DIR, env=test_environment, check=True)
print('Offline gate, cache, statistics and reporting checks passed.')


Offline gate, cache, statistics and reporting checks passed.


## Execute the registered study

The first phase builds E5 indexes and captures complete candidate scores once per question. The second phase replays the same scores under all seven conditions. Logical gate-call counts are recorded; the probe itself scores candidates for every question and is not a latency benchmark.

If the session is interrupted, rerun with the same source and runtime. Completed captures are reused from Drive. Changes to model, input or runtime fingerprints are rejected. A completed valid result is verified and retained without new inference.


In [5]:
from datetime import datetime, timezone
command = [sys.executable, '-u', 'experiments/retriever_ablation/constant_sensitivity/run_sensitivity.py',
           '--output-dir', str(OUTPUT_DIR)]
with CONSOLE_LOG.open('a', encoding='utf-8') as log:
    log.write('\nExecution attempt: ' + datetime.now(timezone.utc).isoformat() + '\n')
    log.flush()
    process = subprocess.Popen(command, cwd=REPO_DIR, env=test_environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    result = process.wait()
if result != 0:
    raise RuntimeError('The study did not finish. Captures and the console log are retained in Drive; review the error before resuming.')


Loading pinned dense: intfloat/e5-small-v2 at ffb93f3bd4047442299a41ebb6fa998a38507c52

Fetching 9 files: 100%|██████████| 9/9 [00:02<00:00,  3.30it/s]
Loading pinned reranker: BAAI/bge-reranker-base at 2cfc18c9415c912f9d8155881c133215df768a70

Fetching 6 files: 100%|██████████| 6/6 [00:03<00:00,  1.92it/s]
Loading Dense Model /root/.cache/huggingface/hub/models--intfloat--e5-small-v2/snapshots/ffb93f3bd4047442299a41ebb6fa998a38507c52 on cuda...

Batches: 100%|██████████| 36/36 [00:02<00:00, 16.88it/s]
Captured rev4/1 (1/136), 0.5s
Captured rev4/2 (2/136), 0.5s
Captured rev4/3 (3/136), 0.5s
Captured rev4/4 (4/136), 0.5s
Captured rev4/5 (5/136), 0.4s
Captured rev4/6 (6/136), 0.4s
Captured rev4/7 (7/136), 0.5s
Captured rev4/8 (8/136), 0.5s
Captured rev4/9 (9/136), 0.5s
Captured rev4/10 (10/136), 0.5s
Captured rev4/11 (11/136), 0.4s
Captured rev4/12 (12/136), 0.4s
Captured rev4/13 (13/136), 0.5s
Captured rev4/14 (14/136), 0.5s
Captured rev4/15 (15/136), 0.5s
Captured rev4/16 (16/136), 0.5

In [6]:
import hashlib
import json
import zipfile

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
assert summary['accepted'] and summary['evaluations'] == 952 and summary['verified_cache_captures'] == 136
assert all(summary['acceptance_checks'].values())
archive_path = OUTPUT_DIR / 'rq1_constant_sensitivity_v1.zip'
with zipfile.ZipFile(archive_path) as bundle:
    assert bundle.testzip() is None
    for line in (OUTPUT_DIR / 'SHA256SUMS').read_text().splitlines():
        expected, relative = line.split('  ', 1)
        assert hashlib.sha256(bundle.read(relative)).hexdigest() == expected, relative
expected, name = (OUTPUT_DIR / 'ARCHIVE_SHA256SUMS').read_text().strip().split('  ', 1)
assert name == archive_path.name
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == expected
print('Verified all 952 evaluations and the complete result archive.')
print('Historical-baseline comparison:', summary['historical_baseline'])
print('ZIP:', archive_path)
print((OUTPUT_DIR / 'SUMMARY.md').read_text())


Verified all 952 evaluations and the complete result archive.
Historical-baseline comparison: {'gate_flags_unchanged': 135, 'gold_rank_unchanged': 134, 'queries': 136, 'top10_unchanged': 65}
ZIP: /content/drive/MyDrive/ComplianceGPT_runs/rq1_constant_sensitivity_v1/rq1_constant_sensitivity_v1.zip
# Primary RQ1 constant sensitivity

Result identity: `rq1_constant_sensitivity_v1`.
Execution source: `a066453a317bba365db707b71745e49fee28811a`.
Protocol SHA-256: `f094327ea5a63f2f25043c1c6f4f0f84bb5bc0f57326bcb374fa4a4c14a6123b`.

Seven conditions share identical inferred channel/cross-encoder scores for each question.
The historical primary notebook logic and frozen rewrites are retained.

## rev4

| Condition | Success@1 | Success@5 | Success@10 | MRR@10 | nDCG@10 | Logical calls | Adopted |
|---|---:|---:|---:|---:|---:|---:|---:|
| baseline | 0.8889 | 1.0000 | 1.0000 | 0.9444 | 0.9590 | 34 | 30 |
| alpha_minus20 | 0.9167 | 1.0000 | 1.0000 | 0.9583 | 0.9692 | 34 | 31 |
| alpha_plus20 | 0.

In [7]:
from google.colab import files
files.download(str(archive_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation

Return `rq1_constant_sensitivity_v1.zip` for independent artifact checks and dissertation integration after the remaining experiments. Report every condition and any deterioration. The new baseline is compared with the saved historical ranks; it does not replace them. This local sensitivity study does not remove evaluation-set tuning bias, validate author labels independently or establish held-out generalization.
